<a id="inizio"></a>
# Esercitazione 3 · Soluzioni

⏱ ~20 min &nbsp;·&nbsp; Dati: `../Dati/U.S. Electricity Prices.csv`, `../Dati/load_total_north_hourly_2024.xlsx`, `../Dati/TexasTurbine.csv`

**In questo notebook impariamo a**

- filtrare e raggruppare i prezzi dell'elettricità per stato
- portare il carico Terna da quartorario a giornaliero con `resample`
- disegnare un istogramma con Plotly Express

<a id="sez-1"></a>
## 1. Domande

Cinque domande brevi: rispondi in una riga, a parole o provando in una cella di codice.

1. Su un DataFrame con indice `0, 1, 2, 3, ...`, quante righe restituiscono `df.loc[0:2]` e `df.iloc[0:2]`? Perché?
2. Cosa restituisce `df.groupby("Categoria")["Vendite"].sum()`? E `df.groupby("Categoria")` da solo?
3. Cosa fa `serie.resample("D").mean()` su una serie con un valore ogni 15 minuti? Che cosa serve perché funzioni?
4. A cosa serve `dayfirst=True` in `pd.to_datetime`? Fai l'esempio di `"01/02/2025"`.
5. Quando serve `how="left"` in `pd.merge`, invece del valore predefinito?

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Risposte**

1. `loc` usa le etichette e include l'ultima: 3 righe. `iloc` usa le posizioni e la esclude, come le liste: 2 righe.
2. Una Series: indice le categorie, valori le somme. `groupby` da solo dà un oggetto GroupBy, che aspetta una funzione.
3. Raggruppa per giorno e fa la media: 96 valori quartorari diventano uno. Serve un indice di date (`set_index`).
4. Nelle date ambigue il primo numero è il giorno: `"01/02/2025"` diventa il 1° febbraio invece del 2 gennaio.
5. Quando vogliamo tenere tutte le righe di sinistra, anche senza corrispondenza: le colonne mancanti diventano
   `NaN`. Con il predefinito `how="inner"` quelle righe spariscono.

</div>

<a id="sez-2"></a>
## 2. Esercizi

Leggiamo i prezzi mensili dell'elettricità negli Stati Uniti, che serve al primo esercizio.
Il prezzo `price` è in centesimi di dollaro per kWh; la colonna `stateDescription` contiene anche
regioni e il totale nazionale (`U.S. Total`).

In [ ]:
import pandas as pd
import plotly.express as px

prezzi = pd.read_csv("../Dati/U.S. Electricity Prices.csv")
prezzi["date"] = pd.to_datetime(prezzi["date"])
prezzi.head()

<a id="es-e3-1"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio E3.1 · Il prezzo residenziale per stato**

Vogliamo sapere in quali stati l'elettricità per le famiglie è costata di più nel 2023.

**Cosa fare.**

1. Metti in `residenziale_2023` le righe di `prezzi` con `sectorName` uguale a `"residential"` e anno 2023.
2. Calcola `prezzo_medio`: il prezzo medio per `stateDescription`, ordinato dal più alto al più basso.

*Suggerimento:* due condizioni tra parentesi unite con `&`; l'anno si legge con `.dt.year`.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione E3.1**

</div>

In [ ]:
residenziale = prezzi["sectorName"] == "residential"
anno_2023 = prezzi["date"].dt.year == 2023
residenziale_2023 = prezzi[residenziale & anno_2023]

prezzo_medio = residenziale_2023.groupby("stateDescription")["price"].mean().sort_values(ascending=False)
prezzo_medio.head()

In [ ]:
from corso import verifica
verifica("E3.1")

<a id="es-e3-2"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio E3.2 · Il carico giornaliero di ottobre**

Il file Terna ha un valore di carico ogni 15 minuti (MW), dal più recente al più vecchio, con quattro orari
ripetuti il 27 ottobre per il ritorno all'ora solare.

**Cosa fare.**

1. Ordina `carico` per `Date`, togli i duplicati di `Date` e metti `Date` come indice.
2. Calcola `giornaliero`: la media giornaliera di `Total Load [MW]` per ottobre 2024.
3. Metti in `giorno_max` il giorno con il carico medio più alto.

*Suggerimento:* `drop_duplicates(subset="Date")`; poi `.loc["2024-10", "Total Load [MW]"]` e `resample("D")`.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione E3.2**

</div>

In [ ]:
carico = pd.read_excel("../Dati/load_total_north_hourly_2024.xlsx")
carico = carico.sort_values("Date").drop_duplicates(subset="Date").set_index("Date")

giornaliero = carico.loc["2024-10", "Total Load [MW]"].resample("D").mean()
giorno_max = giornaliero.idxmax()
giorno_max

In [ ]:
from corso import verifica
verifica("E3.2")

<a id="es-e3-3"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio E3.3 · L'istogramma della potenza della turbina**

Il file della turbina texana ha un valore di potenza per ogni ora dell'anno.

**Cosa fare.**

1. Disegna in `fig` l'istogramma della colonna `System power generated | (kW)` con `px.histogram`.
2. Metti in `ore_ferme` il numero di ore in cui la potenza è 0.

*Suggerimento:* una maschera booleana con `== 0` e `.sum()` conta i `True`.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione E3.3**

</div>

In [ ]:
turbina = pd.read_csv("../Dati/TexasTurbine.csv")

fig = px.histogram(turbina, x="System power generated | (kW)", nbins=30,
                   title="Potenza oraria della turbina")
fig.show()

ore_ferme = (turbina["System power generated | (kW)"] == 0).sum()
ore_ferme

In [ ]:
from corso import verifica
verifica("E3.3")

---

**Fine dell'esercitazione 3.** Precedente: [09 · Plotly per serie storiche](09_Plotly.ipynb) &nbsp;·&nbsp; Prossimo: [10 · Agenti per il coding](10_Agenti_per_il_coding.ipynb)

<sub>Blocco 3 · Analisi dei dati con pandas e visualizzazione · Giornata 2 · Aula Base</sub>